<a href="https://colab.research.google.com/github/alaatababi125/SportTrackGirl4/blob/main/TP1_ETL_Ventes_Etudiant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 TP 1 — Mise en œuvre d'un processus ETL
### Fascicule de travaux pratiques

**Institut Supérieur des Études Technologiques** · Licence en Technologies de l'Informatique — **DSI** · Semestre 5
**Atelier Base de Données Avancée**

| | |
|---|---|
| ⏱️ **Durée** | 3 heures |
| 🎯 **Thème** | Extraire — Transformer — Charger |
| 📦 **Projet support** | Analyse des ventes d'une entreprise |
| 🛠️ **Outils** | Google Colab · Python 3 · `pandas` · `sqlite3` |

> 💡 **Comment utiliser ce notebook dans Colab**
> 1. Menu **Fichier → Enregistrer une copie dans Drive** (pour garder votre propre version).
> 2. Exécutez les cellules **dans l'ordre** (`Maj + Entrée`).
> 3. Les cellules marquées **`# TODO`** sont à compléter. Les cellules **✍️ Votre réponse** sont à rédiger.
> 4. À la fin, téléchargez votre notebook (**Fichier → Télécharger → .ipynb**) et les fichiers produits.

---
## 1. Positionnement de l'atelier
Ce premier TP est le point de départ de l'Atelier Base de Données Avancée. Il introduit la mise en œuvre pratique d'un processus **ETL** (*Extract, Transform, Load*) :

1. **Extraire** les données depuis une ou plusieurs sources ;
2. **Transformer** ces données pour assurer leur qualité et leur homogénéité ;
3. **Charger** le résultat dans une zone cible.

## 2. Objectifs pédagogiques
À l'issue du TP, vous serez capable de :
- comprendre le rôle d'un processus ETL dans un système décisionnel ;
- identifier les sources de données et leurs caractéristiques ;
- extraire des données depuis des fichiers structurés (CSV) ;
- nettoyer et transformer : formats, valeurs nulles, doublons, normalisation de valeurs ;
- charger les données transformées dans une base relationnelle cible ;
- contrôler le chargement et vérifier la qualité des données ;
- documenter le flux ETL et interpréter les erreurs éventuelles.

## 3. Prérequis
- Bases de données relationnelles : tables, attributs, clés primaires et étrangères.
- SQL élémentaire (`SELECT`, `GROUP BY`, `SUM`, `COUNT`).
- Manipulation de fichiers CSV.
- Notions générales de BI (souhaitables).
- Python de base (variables, fonctions) — les commandes `pandas` utiles vous sont rappelées en indice.

## 4. Organisation du TP
| Étape | Activité | Durée | Parties du notebook |
|:-:|---|:-:|:-:|
| 1 | Prise en main du scénario et inspection des sources | 25 min | A |
| 2 | Extraction des données | 30 min | B |
| 3 | Transformation et nettoyage | 55 min | C |
| 4 | Contrôles qualité et chargement dans la base cible | 40 min | D, E |
| 5 | Contrôles après chargement et validation | 20 min | F |
| 6 | Questions de synthèse | 10 min | Section 9 |

*(Une partie **G — bonus** d'automatisation est proposée aux plus rapides.)*

## 5. Scénario professionnel
Une entreprise commerciale dispose de données de ventes provenant de plusieurs services. Les données historiques sont stockées dans des **fichiers CSV**. Avant de pouvoir les exploiter dans un environnement décisionnel, elles doivent être intégrées dans une **base de données cible**.

**Votre mission :** construire un premier flux ETL qui récupère les ventes, corrige les anomalies simples et charge un jeu de données propre dans la base cible.

## 6. Architecture du processus ETL
```
[ Sources CSV ] → [ Extraction ] → [ Zone de transformation ] → [ Contrôles ] → [ Base cible ]
   ventes.csv      pd.read_csv        pandas (DataFrame)          assertions      SQLite : ventes_etl
                                                                   + rejets         (table cible)
```
> ℹ️ Dans ce TP, on **ne construit pas** le modèle multidimensionnel (tables de faits et dimensions) : il sera traité dans l'atelier consacré au modèle multidimensionnel. Ici, l'objectif est de **maîtriser la chaîne ETL elle-même**.

---
## ⚙️ Préparation de l'environnement

In [ ]:
import re
import sqlite3
from datetime import datetime

import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
print("pandas", pd.__version__, "| sqlite", sqlite3.sqlite_version)

pandas 2.2.3 | sqlite 3.45.1


---
## 7. Jeu de données
Le fichier `ventes.csv` contient 8 colonnes. **Des anomalies ont été introduites volontairement** :

| Colonne | Type logique | Exemple | Problème volontaire |
|---|---|---|---|
| `id_vente` | Entier | 1001 | Un même `id_vente` apparaît **deux fois** (doublon) |
| `date_vente` | Date | 2026-09-01 | Plusieurs **formats de date** différents |
| `client` | Texte | Sami Ben Ali | **Espaces** superflus |
| `ville` | Texte | Tunis | Variantes de **casse** |
| `produit` | Texte | Laptop | — |
| `categorie` | Texte | Informatique | Variantes d'écriture (casse, espaces) |
| `quantite` | Entier | 2 | **Valeur manquante** sur une ligne |
| `prix_unitaire` | Décimal | 1250.00 | **Virgule décimale** à contrôler |

▶️ Exécutez la cellule suivante pour **créer le fichier source** dans l'environnement Colab.

In [ ]:
%%writefile ventes.csv
id_vente,date_vente,client,ville,produit,categorie,quantite,prix_unitaire
1001,2026-09-01," Sami Ben Ali ",tunis,Laptop,Informatique,2,1250.00
1002,01/09/2026,"Amel Trabelsi",Tunis,Smartphone,informatique,1,890.00
1003,2026-09-02,"Hatem Jaziri",SFAX,Imprimante,Informatique,,420.00
1004,2026-09-02,"Sonia Ben Salem",sfax,Chaise,Bureautique,3,180.50
1005,2026-09-03,"Mohamed Gharbi",Tunis,Clavier," Bureautique ",2,"75,50"
1006,2026-09-03," Ines Ayari ",tUnIs,Souris,Informatique,1,45.00
1007,04-09-2026,"Karim Trabelsi",Sfax,Écran," informatique ",2,650.00
1008,2026-09-05,"Nadia Ben Amor",Tunis,Laptop,Informatique,1,1250.00
1008,2026-09-05,"Nadia Ben Amor",Tunis,Laptop,Informatique,1,1250.00
1009,2026-09-06,"Walid Jlassi",Sfax,Imprimante,Informatique,2,420.00
1010,2026-09-07,"Sarra Mansour",Tunis,Chaise,bureautique,4,180.50

Writing ventes.csv


---
# 8. Travail demandé

## 🅰️ Partie A — Analyse de la source *(Étape 1 · 25 min)*

**A1.** Identifier le format et la structure du fichier : observez le fichier **brut**, ligne par ligne, tel qu'il est stocké.

In [ ]:
with open("ventes.csv", encoding="utf-8") as f:
    for numero, ligne in enumerate(f, start=1):
        print(f"{numero:>2} | {ligne.rstrip()}")

 1 | id_vente,date_vente,client,ville,produit,categorie,quantite,prix_unitaire
 2 | 1001,2026-09-01," Sami Ben Ali ",tunis,Laptop,Informatique,2,1250.00
 3 | 1002,01/09/2026,"Amel Trabelsi",Tunis,Smartphone,informatique,1,890.00
 4 | 1003,2026-09-02,"Hatem Jaziri",SFAX,Imprimante,Informatique,,420.00
 5 | 1004,2026-09-02,"Sonia Ben Salem",sfax,Chaise,Bureautique,3,180.50
 6 | 1005,2026-09-03,"Mohamed Gharbi",Tunis,Clavier," Bureautique ",2,"75,50"
 7 | 1006,2026-09-03," Ines Ayari ",tUnIs,Souris,Informatique,1,45.00
 8 | 1007,04-09-2026,"Karim Trabelsi",Sfax,Écran," informatique ",2,650.00
 9 | 1008,2026-09-05,"Nadia Ben Amor",Tunis,Laptop,Informatique,1,1250.00
10 | 1008,2026-09-05,"Nadia Ben Amor",Tunis,Laptop,Informatique,1,1250.00
11 | 1009,2026-09-06,"Walid Jlassi",Sfax,Imprimante,Informatique,2,420.00
12 | 1010,2026-09-07,"Sarra Mansour",Tunis,Chaise,bureautique,4,180.50


**A2.** Lister les colonnes et préciser leur type logique. **A3.** Repérer les valeurs manquantes.

In [ ]:
df_diag = pd.read_csv("ventes.csv", dtype=str)

print("Dimensions :", df_diag.shape)

# A3 — valeurs manquantes
print("\nValeurs manquantes par colonne :")
print(df_diag.isna().sum())

print("\nLignes contenant des valeurs manquantes :")
display(df_diag[df_diag.isna().any(axis=1)])

Dimensions : (11, 8)

Valeurs manquantes par colonne :
id_vente         0
date_vente       0
client           0
ville            0
produit          0
categorie        0
quantite         1
prix_unitaire    0
dtype: int64

Lignes contenant des valeurs manquantes :


,id_vente,date_vente,client,ville,produit,categorie,quantite,prix_unitaire
2,1003,2026-09-02,Hatem Jaziri,SFAX,Imprimante,Informatique,NaN,420.00


✍️ **Votre réponse (A2)** — complétez le tableau :

| Colonne | Type logique | Type pandas cible | Type SQL cible |
|---|---|---|---|
| `id_vente` | Int64| | |
| `date_vente` |datetime64[ns] | | |
| `client` |string | | |
| `ville` |srting | | |
| `produit` |string | | |
| `categorie` |string | | |
| `quantite` | Int64| | |
| `prix_unitaire` | float64| | |

**A4.** Repérer les différences de format de date. **A5.** Repérer les différences d'écriture des villes et des catégories.

In [ ]:
# A4 — Identifier les formats de date

def forme_date(s):
    if pd.isna(s):
        return "MANQUANTE"

    s = str(s).strip()

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        return "AAAA-MM-JJ"
    elif re.fullmatch(r"\d{2}/\d{2}/\d{4}", s):
        return "JJ/MM/AAAA"
    elif re.fullmatch(r"\d{2}-\d{2}-\d{4}", s):
        return "JJ-MM-AAAA"
    else:
        return "AUTRE"


print("Formats de date :")
print(df_diag["date_vente"].map(forme_date).value_counts())


# A5 — Différences d'écriture des villes et catégories

print("\nVilles distinctes :")
print(df_diag["ville"].map(repr).value_counts())

print("\nCatégories distinctes :")
print(df_diag["categorie"].map(repr).value_counts())

Formats de date :
date_vente
AAAA-MM-JJ    9
JJ/MM/AAAA    1
JJ-MM-AAAA    1
Name: count, dtype: int64

Villes distinctes :
ville
'Tunis'    5
'Sfax'     2
'tunis'    1
'SFAX'     1
'sfax'     1
'tUnIs'    1
Name: count, dtype: int64

Catégories distinctes :
categorie
'Informatique'      6
'informatique'      1
'Bureautique'       1
' Bureautique '     1
' informatique '    1
'bureautique'       1
Name: count, dtype: int64


✍️ **Votre réponse (A4/A5)** — combien de formats de date ? Combien de villes et de catégories *réelles* ? Quelle ambiguïté pose la date `01/09/2026` ?

**A6.** Identifier les risques que ces anomalies peuvent créer lors d'une analyse ultérieure.

Pour vous aider, mesurons l'impact **si on charge les données sans les nettoyer** :

In [ ]:
print("Nombre de villes distinctes brutes :",
      df_diag["ville"].nunique())

print("Nombre de catégories distinctes brutes :",
      df_diag["categorie"].nunique())

print("Nombre de doublons sur id_vente :",
      df_diag.duplicated(subset=["id_vente"]).sum())

print("Nombre de valeurs manquantes :")
print(df_diag.isna().sum())

prix_test = pd.to_numeric(
    df_diag["prix_unitaire"].str.replace(",", ".", regex=False),
    errors="coerce"
)

print("Prix non convertibles directement :",
      prix_test.isna().sum())

print("Formats de date différents :",
      df_diag["date_vente"].map(forme_date).nunique())

Nombre de villes distinctes brutes : 6
Nombre de catégories distinctes brutes : 6
Nombre de doublons sur id_vente : 1
Nombre de valeurs manquantes :
id_vente         0
date_vente       0
client           0
ville            0
produit          0
categorie        0
quantite         1
prix_unitaire    0
dtype: int64
Prix non convertibles directement : 0
Formats de date différents : 3


✍️ **Votre réponse (A6)** — pour chaque anomalie, citez un risque concret pour une analyse décisionnelle.

---
## 🅱️ Partie B — Extraction *(Étape 2 · 30 min)*
Mettre en place un flux de lecture de `ventes.csv`.

**B1.** Configurer la connexion à la source CSV. **B2.** Définir séparateur et encodage. **B3.** Importer les colonnes avec leurs types appropriés. **B4.** Visualiser les premières lignes. **B5.** Vérifier le nombre de lignes lues.

> 🔑 **Principe :** à l'extraction, on lit **tout en texte** (`dtype=str`) puis on convertit **explicitement** pendant la transformation. Ainsi, une valeur invalide (`"75,50"`) ne provoque pas d'erreur silencieuse et reste **traçable**.

In [ ]:
# B1 — connexion à la source
CHEMIN_SOURCE = "ventes.csv"

# B2 — séparateur et encodage
SEPARATEUR = ","
ENCODAGE = "utf-8"

# B3 — extraction
def extraire(chemin=CHEMIN_SOURCE):
    return pd.read_csv(
        chemin,
        sep=SEPARATEUR,
        encoding=ENCODAGE,
        dtype=str
    )

df_raw = extraire()

# B4 — afficher les 5 premières lignes
print("Premières lignes :")
display(df_raw.head())

# B5 — vérifier le nombre de lignes
assert len(df_raw) == 11

print("Nombre de lignes lues :", len(df_raw))
print("✅ Extraction correcte")

Premières lignes :


,id_vente,date_vente,client,ville,produit,categorie,quantite,prix_unitaire
0,1001,2026-09-01,Sami Ben Ali,tunis,Laptop,Informatique,2,1250.00
1,1002,01/09/2026,Amel Trabelsi,Tunis,Smartphone,informatique,1,890.00
2,1003,2026-09-02,Hatem Jaziri,SFAX,Imprimante,Informatique,NaN,420.00
3,1004,2026-09-02,Sonia Ben Salem,sfax,Chaise,Bureautique,3,180.50
4,1005,2026-09-03,Mohamed Gharbi,Tunis,Clavier,Bureautique,2,"75,50"


Nombre de lignes lues : 11
✅ Extraction correcte


✍️ **Votre réponse (B3)** — pourquoi lit-on toutes les colonnes en texte (`dtype=str`) à l'extraction ?

n lit toutes les colonnes en texte afin d'éviter qu'une conversion automatique provoque une perte ou une mauvaise interprétation des données. Les conversions sont réalisées explicitement pendant la transformation, ce qui permet de détecter et de tracer les valeurs incorrectes comme 75,50

---
## 🅲 Partie C — Transformation *(Étape 3 · 55 min)*
Appliquer les transformations suivantes **sur une copie** des données brutes (`df_raw` reste intact pour la traçabilité) :

| N° | Transformation |
|:-:|---|
| C1 | Supprimer les espaces inutiles dans `client` |
| C2 | Uniformiser les villes (`Tunis`, `Sfax`) |
| C3 | Uniformiser les catégories (`Informatique`, `Bureautique`) |
| C4 | Convertir `date_vente` vers un format de date unique |
| C5 | Convertir `prix_unitaire` en type numérique |
| C6 | Traiter les quantités manquantes selon **une règle explicitement justifiée** |
| C7 | Traiter les doublons *(ajouté : l'anomalie existe dans la source)* |
| C8 | Créer la colonne `montant = quantite × prix_unitaire` |

> 🧾 **Journal ETL :** à chaque étape, on note le nombre de lignes avant/après. C'est la base du bilan demandé en D.

In [ ]:
journal = []

def noter(etape, regle, avant, apres):
    journal.append({
        "étape": etape,
        "règle appliquée": regle,
        "lignes avant": avant,
        "lignes après": apres
    })


df = df_raw.copy()
n = len(df)


# C1 — supprimer les espaces superflus dans client
df["client"] = (
    df["client"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

noter(
    "C1",
    "client : suppression des espaces superflus",
    n,
    len(df)
)


# C2 — uniformiser les villes
referentiel_villes = {
    "tunis": "Tunis",
    "sfax": "Sfax"
}

df["ville"] = (
    df["ville"]
    .str.strip()
    .str.lower()
    .map(referentiel_villes)
)

noter(
    "C2",
    "ville : normalisation de la casse",
    n,
    len(df)
)


# C3 — uniformiser les catégories
referentiel_categories = {
    "informatique": "Informatique",
    "bureautique": "Bureautique"
}

df["categorie"] = (
    df["categorie"]
    .str.strip()
    .str.lower()
    .map(referentiel_categories)
)

noter(
    "C3",
    "categorie : normalisation de la casse et des espaces",
    n,
    len(df)
)


# C4 — uniformiser les dates

def parse_date(s):
    s = str(s).strip()

    for format_date in [
        "%Y-%m-%d",
        "%d/%m/%Y",
        "%d-%m-%Y"
    ]:
        try:
            return datetime.strptime(s, format_date)
        except ValueError:
            pass

    return pd.NaT


df["date_vente"] = df["date_vente"].map(parse_date)

noter(
    "C4",
    "date : conversion vers un format de date unique",
    n,
    len(df)
)


# C5 — convertir les prix en numérique
df["prix_unitaire"] = pd.to_numeric(
    df["prix_unitaire"]
    .str.replace(",", ".", regex=False),
    errors="coerce"
)

noter(
    "C5",
    "prix_unitaire : conversion en numérique",
    n,
    len(df)
)


# Préparation des types
df["quantite"] = pd.to_numeric(
    df["quantite"],
    errors="coerce"
).astype("Int64")

df["id_vente"] = pd.to_numeric(
    df["id_vente"],
    errors="coerce"
).astype("Int64")


display(df)
print(df.dtypes)

,id_vente,date_vente,client,ville,produit,categorie,quantite,prix_unitaire
0,1001,2026-09-01,Sami Ben Ali,Tunis,Laptop,Informatique,2,1250.0
1,1002,2026-09-01,Amel Trabelsi,Tunis,Smartphone,Informatique,1,890.0
2,1003,2026-09-02,Hatem Jaziri,Sfax,Imprimante,Informatique,<NA>,420.0
3,1004,2026-09-02,Sonia Ben Salem,Sfax,Chaise,Bureautique,3,180.5
4,1005,2026-09-03,Mohamed Gharbi,Tunis,Clavier,Bureautique,2,75.5
5,1006,2026-09-03,Ines Ayari,Tunis,Souris,Informatique,1,45.0
6,1007,2026-09-04,Karim Trabelsi,Sfax,Écran,Informatique,2,650.0
7,1008,2026-09-05,Nadia Ben Amor,Tunis,Laptop,Informatique,1,1250.0
8,1008,2026-09-05,Nadia Ben Amor,Tunis,Laptop,Informatique,1,1250.0
9,1009,2026-09-06,Walid Jlassi,Sfax,Imprimante,Informatique,2,420.0


id_vente                  Int64
date_vente       datetime64[ns]
client                   object
ville                    object
produit                  object
categorie                object
quantite                  Int64
prix_unitaire           float64
dtype: object


In [ ]:
# C8 — montant = quantite × prix_unitaire

df["montant"] = (
    df["quantite"].astype(float) *
    df["prix_unitaire"]
).round(2)

noter(
    "C8",
    "montant = quantite x prix_unitaire",
    len(df),
    len(df)
)

df_ok = df.sort_values("id_vente").reset_index(drop=True)

display(df_ok)

,id_vente,date_vente,client,ville,produit,categorie,quantite,prix_unitaire,montant
0,1001,2026-09-01,Sami Ben Ali,Tunis,Laptop,Informatique,2,1250.0,2500.0
1,1002,2026-09-01,Amel Trabelsi,Tunis,Smartphone,Informatique,1,890.0,890.0
2,1004,2026-09-02,Sonia Ben Salem,Sfax,Chaise,Bureautique,3,180.5,541.5
3,1005,2026-09-03,Mohamed Gharbi,Tunis,Clavier,Bureautique,2,75.5,151.0
4,1006,2026-09-03,Ines Ayari,Tunis,Souris,Informatique,1,45.0,45.0
5,1007,2026-09-04,Karim Trabelsi,Sfax,Écran,Informatique,2,650.0,1300.0
6,1008,2026-09-05,Nadia Ben Amor,Tunis,Laptop,Informatique,1,1250.0,1250.0
7,1009,2026-09-06,Walid Jlassi,Sfax,Imprimante,Informatique,2,420.0,840.0
8,1010,2026-09-07,Sarra Mansour,Tunis,Chaise,Bureautique,4,180.5,722.0


In [ ]:
rejets = []   # trace de tout ce qui est écarté : liste de dictionnaires {..valeurs brutes.., "type": ..., "motif": ...}

# C7 — doublons. Indices : df.duplicated(subset=["id_vente"], keep="first")
#      Avant de supprimer : les doublons sont-ils identiques (df.duplicated()) ?
#      N'oubliez pas d'ajouter la ligne écartée à `rejets` (valeurs brutes : df_raw.loc[i].to_dict())
# TODO

# C6 — quantités manquantes : quelle règle ? (à JUSTIFIER dans la cellule de réponse ci-dessous)
# TODO

✍️ **Votre réponse (C6)** — quelle règle avez-vous appliquée pour la quantité manquante de la vente 1003 ? **Justifiez** et expliquez pourquoi les autres options (0, 1, moyenne, NULL) sont risquées.

In [ ]:
rejets = []


# C7 — doublons
masque_doublons = df.duplicated(
    subset=["id_vente"],
    keep="first"
)

# Vérifier si les doublons sont totalement identiques
print("Doublons totalement identiques :")
display(df[df.duplicated(keep=False)])

# Ajouter les lignes doublons aux rejets
for i in df.index[masque_doublons]:
    rejets.append({
        **df_raw.loc[i].to_dict(),
        "type": "doublon",
        "motif": "id_vente déjà présent"
    })

# Supprimer les doublons
df = df[~masque_doublons].copy()

noter(
    "C7",
    "suppression des doublons sur id_vente",
    len(df) + masque_doublons.sum(),
    len(df)
)


# C6 — quantité manquante
masque_quantite = df["quantite"].isna()

for i in df.index[masque_quantite]:
    rejets.append({
        **df_raw.loc[i].to_dict(),
        "type": "quantité manquante",
        "motif": "quantité absente et impossible à déterminer"
    })

# Rejeter la ligne dont la quantité est manquante
df = df[~masque_quantite].copy()

noter(
    "C6",
    "rejet des lignes dont la quantité est manquante",
    len(df) + masque_quantite.sum(),
    len(df)
)

print("Nombre de lignes après traitement :", len(df))

Doublons totalement identiques :


,id_vente,date_vente,client,ville,produit,categorie,quantite,prix_unitaire
7,1008,2026-09-05,Nadia Ben Amor,Tunis,Laptop,Informatique,1,1250.0
8,1008,2026-09-05,Nadia Ben Amor,Tunis,Laptop,Informatique,1,1250.0


Nombre de lignes après traitement : 9


---
## 🅳 Partie D — Contrôle de qualité *(Étape 4 · avant chargement)*
**D1.** Compter les lignes avant/après. **D2.** Aucune quantité invalide. **D3.** Prix numériques et positifs. **D4.** Dates conformes au format choisi. **D5.** Identifier les lignes rejetées. **D6.** Produire un court bilan des anomalies et des règles appliquées.

In [ ]:
def verifier(libelle, condition):
    print(("✅" if condition else "❌"), libelle)
    return bool(condition)


n_source = len(df_raw)

# D1
n_doublons = sum(
    1 for r in rejets
    if r["type"] == "doublon"
)

n_rejets = sum(
    1 for r in rejets
    if r["type"] != "doublon"
)

n_charge = len(df_ok)

print("Lignes source :", n_source)
print("Doublons :", n_doublons)
print("Rejets :", n_rejets)
print("Lignes chargées :", n_charge)


resultats = [

    # D2
    verifier(
        "D2 — aucune quantité NULL ou <= 0",
        not (
            df_ok["quantite"].isna() |
            (df_ok["quantite"] <= 0)
        ).any()
    ),

    # D3
    verifier(
        "D3 — prix numériques et strictement positifs",
        pd.api.types.is_numeric_dtype(df_ok["prix_unitaire"])
        and
        (df_ok["prix_unitaire"] > 0).all()
    ),

    # D4
    verifier(
        "D4 — dates valides",
        df_ok["date_vente"].notna().all()
    ),

    # ID unique
    verifier(
        "ID vente unique",
        df_ok["id_vente"].is_unique
    ),

    # Villes valides
    verifier(
        "Villes valides",
        set(df_ok["ville"]).issubset({"Tunis", "Sfax"})
    ),

    # Catégories valides
    verifier(
        "Catégories valides",
        set(df_ok["categorie"]).issubset(
            {"Informatique", "Bureautique"}
        )
    ),

    # Réconciliation
    verifier(
        "Réconciliation source = chargées + doublons + rejets",
        n_source == n_charge + n_doublons + n_rejets
    )
]

assert all(resultats), "Des contrôles ont échoué : ne PAS charger !"

Lignes source : 11
Doublons : 1
Rejets : 1
Lignes chargées : 9
✅ D2 — aucune quantité NULL ou <= 0
✅ D3 — prix numériques et strictement positifs
✅ D4 — dates valides
✅ ID vente unique
✅ Villes valides
✅ Catégories valides
✅ Réconciliation source = chargées + doublons + rejets


In [ ]:
# D5 — afficher les lignes rejetées
df_rejets = pd.DataFrame(rejets)

print("Lignes rejetées :")
display(df_rejets)


# D6 — afficher le journal ETL
df_journal = pd.DataFrame(journal)

print("Journal des règles appliquées :")
display(df_journal)


# Sauvegarder les livrables
df_rejets.to_csv(
    "rejets_ventes.csv",
    index=False,
    encoding="utf-8"
)

df_journal.to_csv(
    "journal_etl.csv",
    index=False,
    encoding="utf-8"
)

df_ok.to_csv(
    "ventes_propres.csv",
    index=False,
    encoding="utf-8"
)

Lignes rejetées :


,id_vente,date_vente,client,ville,produit,categorie,quantite,prix_unitaire,type,motif
0,1008,2026-09-05,Nadia Ben Amor,Tunis,Laptop,Informatique,1,1250.00,doublon,id_vente déjà présent
1,1003,2026-09-02,Hatem Jaziri,SFAX,Imprimante,Informatique,NaN,420.00,quantité manquante,quantité absente et impossible à déterminer


Journal des règles appliquées :


,étape,règle appliquée,lignes avant,lignes après
0,C1,client : suppression des espaces superflus,11,11
1,C2,ville : normalisation de la casse,11,11
2,C3,categorie : normalisation de la casse et des e...,11,11
3,C4,date : conversion vers un format de date unique,11,11
4,C5,prix_unitaire : conversion en numérique,11,11
5,C7,suppression des doublons sur id_vente,11,10
6,C6,rejet des lignes dont la quantité est manquante,10,9
7,C8,montant = quantite x prix_unitaire,9,9


In [ ]:
# D5 — afficher les lignes écartées (indice : pd.DataFrame(rejets))
# D6 — afficher le journal des règles appliquées (indice : pd.DataFrame(journal))
# TODO

# Sauvegarde des traces (livrables) : décommentez quand vos DataFrames existent
# df_rejets.to_csv("rejets_ventes.csv", index=False, encoding="utf-8")
# df_journal.to_csv("journal_etl.csv", index=False, encoding="utf-8")
# df_ok.to_csv("ventes_propres.csv", index=False, encoding="utf-8")

✍️ **Votre bilan (D6)** — rédigez un court tableau : anomalie détectée → règle appliquée → nombre de lignes concernées.

---
## 🅴 Partie E — Chargement *(Étape 4 · 40 min)*
Créer la table cible `ventes_etl` dans une base relationnelle, puis y charger les données transformées.

> 🗄️ Nous utilisons **SQLite** (intégré à Python, aucune installation). Le script SQL de création est **identique** pour MySQL ou PostgreSQL.

In [ ]:
DDL = '''
CREATE TABLE ventes_etl (
    id_vente       INTEGER PRIMARY KEY,
    date_vente     DATE,
    client         VARCHAR(100),
    ville          VARCHAR(50),
    produit        VARCHAR(100),
    categorie      VARCHAR(80),
    quantite       INTEGER,
    prix_unitaire  DECIMAL(12,2),
    montant        DECIMAL(14,2)
);
'''

con = sqlite3.connect("entrepot_ventes.db")
cur = con.cursor()


# E1 — supprimer puis recréer la table
cur.execute("DROP TABLE IF EXISTS ventes_etl")
cur.execute(DDL)


# E2 — préparer les tuples ligne native ( int , str, float)
lignes = [
    (
        int(r.id_vente),
        r.date_vente.strftime("%Y-%m-%d"),
        str(r.client),
        str(r.ville),
        str(r.produit),
        str(r.categorie),
        int(r.quantite),
        float(r.prix_unitaire),
        float(r.montant)
    )
    for r in df_ok.itertuples(index=False)
]


# E3 — insertion
cur.executemany(
    """
    INSERT INTO ventes_etl
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """,
    lignes
)

con.commit() # Sauvegarder


# E4 — contrôle
nb_charge = cur.execute(
    "SELECT COUNT(*) FROM ventes_etl" # bech te7seb les ligne
).fetchone()[0] # 9 nombre de ligne

print("Nombre de lignes chargées :", nb_charge)

assert nb_charge == len(df_ok) # assert bech ntastou

print("✅ Chargement réussi")

Nombre de lignes chargées : 9
✅ Chargement réussi


✍️ **Votre réponse (E)** — pourquoi utilise-t-on des paramètres `?` plutôt que de construire les `INSERT` par concaténation de texte ?

---
## 🅵 Partie F — Contrôles après chargement *(Étape 5 · 20 min)*
Interrogez **la base** (et non plus le DataFrame) : on vérifie ce qui est *réellement* stocké. Comparez ensuite avec les données sources et **expliquez toute différence**.

In [ ]:
def sql(requete):
    return pd.read_sql_query(requete, con)


# F1
print("F1 — Nombre de lignes")
display(sql("""
    SELECT COUNT(*) AS nombre_lignes
    FROM ventes_etl
"""))


# F2 — CA total
print("F2 — Chiffre d'affaires total")
display(sql("""
    SELECT ROUND(SUM(montant), 2) AS chiffre_affaires
    FROM ventes_etl
"""))


# F3 — CA par ville
print("F3 — Chiffre d'affaires par ville")
display(sql("""
    SELECT ville,
           ROUND(SUM(montant), 2) AS chiffre_affaires
    FROM ventes_etl
    GROUP BY ville
    ORDER BY ville
"""))


# F4 — CA par catégorie
print("F4 — Chiffre d'affaires par catégorie")
display(sql("""
    SELECT categorie,
           ROUND(SUM(montant), 2) AS chiffre_affaires
    FROM ventes_etl
    GROUP BY categorie
    ORDER BY categorie
"""))


# F5 — quantités invalides
print("F5 — Quantités NULL ou <= 0")
display(sql("""
    SELECT *
    FROM ventes_etl
    WHERE quantite IS NULL
       OR quantite <= 0
"""))


# F6 — prix invalides
print("F6 — Prix NULL ou <= 0")
display(sql("""
    SELECT *
    FROM ventes_etl
    WHERE prix_unitaire IS NULL
       OR prix_unitaire <= 0
"""))


# F7 — doublons
print("F7 — ID présents plusieurs fois")
display(sql("""
    SELECT id_vente, COUNT(*) AS nombre
    FROM ventes_etl
    GROUP BY id_vente
    HAVING COUNT(*) > 1
"""))

F1 — Nombre de lignes


,nombre_lignes
0,9


F2 — Chiffre d'affaires total


,chiffre_affaires
0,8239.5


F3 — Chiffre d'affaires par ville


...

,id_vente,date_vente,client,ville,produit,categorie,quantite,prix_unitaire,montant


F6 — Prix NULL ou <= 0


,id_vente,date_vente,client,ville,produit,categorie,quantite,prix_unitaire,montant


F7 — ID présents plusieurs fois


,id_vente,nombre


✍️ **Votre analyse (F)** — comparez avec la source : combien de lignes avez-vous perdues et pourquoi ? Quel aurait été le chiffre d'affaires **sans nettoyage** ? Vérifier uniquement le total du CA suffit-il ? Justifiez.

> 💡 *Astuce :* calculez le « CA naïf » sur `df_raw` en convertissant directement `prix_unitaire` et `quantite` avec `pd.to_numeric(..., errors="coerce")`.

---
## 🅶 Partie G — Bonus : automatiser le flux *(pour les plus rapides)*
Regroupez **extraction → transformation → contrôles → chargement** dans **une seule fonction** rejouable. Un ETL qui tourne chaque jour doit être **automatisé, reproductible et idempotent** (relancer ne duplique rien).

In [ ]:
def etl_ventes(
    chemin_csv="ventes.csv",
    chemin_db="entrepot_ventes.db"
):

    # Extraction
    raw = pd.read_csv(
        chemin_csv,
        sep=",",
        encoding="utf-8",
        dtype=str
    )

    # Transformation
    data = raw.copy()

    data["client"] = (
        data["client"]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    data["ville"] = (
        data["ville"]
        .str.strip()
        .str.lower()
        .map({
            "tunis": "Tunis",
            "sfax": "Sfax"
        })
    )

    data["categorie"] = (
        data["categorie"]
        .str.strip()
        .str.lower()
        .map({
            "informatique": "Informatique",
            "bureautique": "Bureautique"
        })
    )

    data["date_vente"] = data["date_vente"].map(parse_date)

    data["prix_unitaire"] = pd.to_numeric(
        data["prix_unitaire"]
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

    data["quantite"] = pd.to_numeric(
        data["quantite"],
        errors="coerce"
    ).astype("Int64")

    data["id_vente"] = pd.to_numeric(
        data["id_vente"],
        errors="coerce"
    ).astype("Int64")


    # Doublons
    doublons = data.duplicated(
        subset=["id_vente"],
        keep="first"
    ).sum()

    data = data.drop_duplicates(
        subset=["id_vente"],
        keep="first"
    )


    # Quantités manquantes
    rejets = data["quantite"].isna().sum()

    data = data.dropna(
        subset=["quantite"]
    )


    # Montant
    data["montant"] = (
        data["quantite"].astype(float)
        * data["prix_unitaire"]
    ).round(2)


    # Chargement
    db = sqlite3.connect(chemin_db)
    cursor = db.cursor()

    cursor.execute("DROP TABLE IF EXISTS ventes_etl")
    cursor.execute(DDL)

    lignes = [
        (
            int(r.id_vente),
            r.date_vente.strftime("%Y-%m-%d"),
            str(r.client),
            str(r.ville),
            str(r.produit),
            str(r.categorie),
            int(r.quantite),
            float(r.prix_unitaire),
            float(r.montant)
        )
        for r in data.itertuples(index=False)
    ]

    cursor.executemany(
        "INSERT INTO ventes_etl VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
        lignes
    )

    db.commit()

    ca = cursor.execute(
        "SELECT SUM(montant) FROM ventes_etl"
    ).fetchone()[0]

    db.close()

    return {
        "source": len(raw),
        "doublons": doublons,
        "rejets": rejets,
        "charges": len(data),
        "CA": round(ca, 2)
    }


print(etl_ventes())
print(etl_ventes())

{'source': 11, 'doublons': np.int64(1), 'rejets': np.int64(1), 'charges': 9, 'CA': 8239.5}
{'source': 11, 'doublons': np.int64(1), 'rejets': np.int64(1), 'charges': 9, 'CA': 8239.5}


---
# 9. Questions de compréhension *(Étape 6 · 10 min)*

**Question 1.** Quelle est la différence entre un **ETL** et une **simple importation** de données ?

✍️ **Votre réponse :** …

**Question 2.** Pourquoi faut-il **transformer** les données **avant** leur chargement ?

✍️ **Votre réponse :** …

**Question 3.** Quelle est la différence entre une **erreur de format** et une **valeur manquante** ? Donnez un exemple de chacune tiré du TP.

✍️ **Votre réponse :** …

**Question 4.** Pourquoi faut-il **contrôler le nombre de lignes** avant et après un traitement ETL ?

✍️ **Votre réponse :** …

**Question 5.** Que se passe-t-il si une **colonne numérique contient une valeur textuelle** ?

✍️ **Votre réponse :** …

**Question 6.** Quel est l'impact d'un **doublon** sur les résultats d'une analyse ?

✍️ **Votre réponse :** …

**Question 7.** Pourquoi faut-il **conserver une trace des lignes rejetées** ?

✍️ **Votre réponse :** …

**Question 8.** Pourquoi les **règles de transformation** doivent-elles être définies **explicitement et justifiées** ?

✍️ **Votre réponse :** …

**Question 9.** Dans quel cas un processus ETL doit-il être **automatisé** ?

✍️ **Votre réponse :** …

**Question 10.** Quelle différence faites-vous entre les **données sources** et les **données cibles** ?

✍️ **Votre réponse :** …

---
## 10. Travail à remettre
- ☐ Le fichier source utilisé (`ventes.csv`)
- ☐ Le **notebook complété** (flux ETL : extraction, transformation, contrôles, chargement)
- ☐ Le script SQL de création de la table cible (cellule DDL)
- ☐ Le tableau récapitulatif des transformations (`journal_etl.csv` + votre bilan D6)
- ☐ Le compte rendu des contrôles (sorties des parties D et F) et les lignes rejetées (`rejets_ventes.csv`)
- ☐ Les réponses aux **10 questions** de compréhension

In [ ]:
# Téléchargement des fichiers produits (fonctionne dans Google Colab)
try:
    from google.colab import files
    for f in ["ventes.csv", "ventes_propres.csv", "rejets_ventes.csv", "journal_etl.csv"]:
        try:
            files.download(f)
        except Exception as e:
            print("Fichier non disponible :", f)
except ImportError:
    print("Hors Colab : les fichiers sont dans le dossier courant.")

---
## 11. Synthèse
À l'issue de ce TP, vous devez être capable de **décrire et d'implémenter une chaîne ETL simple** : identifier une source, extraire les données, appliquer des **transformations contrôlées**, charger une cible et **vérifier la qualité** du résultat.

```
11 lignes (source) → détection des anomalies → 1 doublon supprimé → 1 ligne rejetée (quantité manquante)
                   → 9 lignes valides → calcul du montant → contrôles qualité → chargement dans ventes_etl
                   → Chiffre d'affaires total = 8 239,50
```
**Suite du programme :** flux de contrôle, puis ETL **incrémental**, conformément à la progression du référentiel.